# 2 — Across cohorts: federated clustering

Day two. Each cohort clustered alone in notebook 1; now they combine **without
any of them sharing sample-level rows**.

Two things pool, by two different rules:

| what pools | rule | gives |
|---|---|---|
| sufficient statistics | **add** four `p×p` matrices | the federated **tree** — exact |
| bootstrap counts | **sum / concatenate** tallies | the federated **AU** |

They are independent. The first is exact: adding statistics reproduces the
distance matrix of the pooled raw data bit for bit, so the federated dendrogram
*is* the centralised dendrogram.

In [ ]:
import warnings
import numpy as np, pandas as pd

from pvclust_py.core import pvclust, count_edges
from pvclust_py.datasets import load_lung, load_cohorts
from pvclust_py.distance import pairwise_stats, distance
from pvclust_py.hclust import edge_table, linkage
from pvclust_py.aggregate import (shared_features, federated_tree, pool_counts,
                                  federated_edges, consensus_clusters)

warnings.simplefilter('ignore')

# Chosen once, and used for the federated tree AND the centralised comparison.
# They must match, or the two are not comparable.
METHOD_DIST = 'minkowski'
METHOD_HCLUST = 'ward.D2'

cohorts = load_cohorts(n_cohorts=3)
{k: v.shape for k, v in cohorts.items()}

## Step 1 — the shared vocabulary

Clustering federates only over objects every cohort measures. For a SomaScan panel
this is nearly free; for RNAseq it means an explicit gene-ID intersection.

In [ ]:
shared = shared_features(df.columns for df in cohorts.values())
print(f'{len(shared)} shared objects')
cohorts = {k: df[shared] for k, df in cohorts.items()}

## Step 2 — what actually leaves each cohort

Four `p×p` matrices, every entry a sum over rows. No row survives individually,
and the payload does **not** grow with the number of samples a cohort holds.

In [ ]:
stats = {name: pairwise_stats(df.to_numpy(float)) for name, df in cohorts.items()}

p = len(shared)
for name, st in stats.items():
    print(f'{name}: n={st["n"]:3d} samples  ->  ships {4*p*p:,} numbers')
print(f'\npayload is 4p^2 = {4*p*p:,} regardless of n. But it is QUADRATIC in p:')
for pp in [100, 1000, 5000, 7000, 20000]:
    print(f'   p={pp:6,} -> {4*pp*pp*8/1e6:9,.0f} MB')
print('\nExact mode is comfortable to ~1-2k objects. Past that, use counts mode,')
print('whose payload is ~p x n_scales rows instead.')

## Step 3 — the federated tree

The aggregator adds the statistics. Nothing else.

In [ ]:
D_fed, Z_fed, catalogue = federated_tree(stats.values(), shared,
                                         method_dist=METHOD_DIST,
                                         method_hclust=METHOD_HCLUST)
print(f'federated tree: {len(catalogue)} clusters')

# the check that justifies calling this 'exact'
full = load_lung()[shared]
D_central = distance(full.to_numpy(float), METHOD_DIST)
print(f'max difference from the centralised distance matrix: '
      f'{np.abs(D_fed - D_central).max():.2e}')

### What each cohort could see alone

This is the number that matters. Federation being *equal* to centralised is the
theorem — it could hardly come out otherwise. The interesting comparison is
against what a cohort sees on its own.

In [ ]:
truth = {e['edge_id'] for e in edge_table(linkage(D_central, METHOD_HCLUST), shared)}

solo = {}
for name, df in cohorts.items():
    Z = linkage(distance(df.to_numpy(float), METHOD_DIST), METHOD_HCLUST)
    solo[name] = {e['edge_id'] for e in edge_table(Z, shared)}
    print(f'{name} alone: recovers {len(solo[name] & truth):2d}/{len(truth)} clusters')

agree = set.intersection(*solo.values())
fed = {e['edge_id'] for e in catalogue}
print(f'\nall three cohorts agree on: {len(agree):2d}/{len(truth)}')
print(f'federated:                  {len(fed & truth):2d}/{len(truth)}')

## Step 4 — federated AU

Now the counts. Each cohort bootstraps against the **federated catalogue** rather
than its own tree — that is what makes the tallies addable. A cohort can count a
cluster its own tree never produced; that is exactly the point.

This is the second pass of counts-mode federation.

In [ ]:
NBOOT = 200   # raise to 1000 (R's default) for real results
members = [e['members'] for e in catalogue]

count_frames = []
for name, df in cohorts.items():
    counts, r_eff, nboot_vec, _ = count_edges(
        df.to_numpy(float), members, shared, nboot=NBOOT, seed=42,
        method_dist=METHOD_DIST, method_hclust=METHOD_HCLUST)
    count_frames.append(pd.DataFrame([
        {'edge_id': e['edge_id'], 'r': float(r_eff[j]), 'n': len(df),
         'nboot': int(nboot_vec[j]), 'count': int(counts[i, j])}
        for i, e in enumerate(catalogue) for j in range(len(r_eff))]))
    print(f'{name}: {len(count_frames[-1]):,} count rows  '
          f'(scales {r_eff.min():.3f}-{r_eff.max():.3f})')

### Scales must be put on a common footing first

Each cohort's `r` is relative to **its own** n. `r = 1.0` at a 55-sample cohort
means 55 rows; at a 17-sample cohort it means 17. Those are different scales, and
adding their counts as though they matched would be wrong.

`pool_counts` re-expresses every measurement against the pooled n:

```
size     = r_cohort x n_cohort        # absolute rows drawn
r_pooled = size / sum(n_cohort)
```

**And here is the open question.** After rescaling, no cohort can reach
`r_pooled = 1` — none of them holds all the rows. So the pooled scatter has more
points across a wider spread, but sits *below* 1, which makes the extrapolation to
σ² = −1 longer than a centralised run would need.

More scales and wider spread help condition the two-parameter fit; a longer
extrapolation hurts. Which wins is **not settled here**, and the printout below is
the evidence you would need to start answering it.

In [ ]:
pooled = pool_counts(count_frames)   # rescales to the pooled n by default

n_total = sum(len(df) for df in cohorts.values())
print(f'pooled n = {n_total}\n')
print('as each cohort sees it (r relative to its own n):')
for f, (name, df) in zip(count_frames, cohorts.items()):
    print(f'  {name} (n={len(df):2d}): r {f.r.min():.3f}-{f.r.max():.3f}, '
          f'absolute {int(f.r.min()*len(df))}-{int(f.r.max()*len(df))} rows')
print('\nafter rescaling to the pooled n:')
print(f'  POOLED: r {pooled.r.min():.3f}-{pooled.r.max():.3f} '
      f'({pooled.r.nunique()} distinct scales vs {count_frames[0].r.nunique()} per cohort)')
print(f'  reaches r=1? {pooled.r.max() >= 1.0}   <-- the open question above')

In [ ]:
fed_edges = federated_edges(pooled, catalogue)
fed_edges[['n_members', 'bp', 'au', 'si', 'se_au', 'pchi', 'n_scales']].head(10)

### Federated AU vs going it alone

In [ ]:
solo_au = {}
for name, df in cohorts.items():
    res = pvclust(df.to_numpy(float), shared, nboot=NBOOT, seed=42,
                  method_dist=METHOD_DIST, method_hclust=METHOD_HCLUST)
    solo_au[name] = {e['edge_id']: e['au'] for e in res.edges}

cmp = fed_edges[['edge_id', 'n_members', 'au']].rename(columns={'au': 'au_federated'})
for name in cohorts:
    cmp[f'au_{name}'] = cmp.edge_id.map(solo_au[name])
cmp.head(10)

**Blank cells are the point.** A missing value means that cohort's own tree never
produced the cluster, so alone it had nothing to report. The federation gives it
an answer.

## Step 5 — the consensus set

In [ ]:
cons = consensus_clusters(fed_edges, alpha=0.95)
print(f'{len(cons)} clusters at federated AU >= 0.95:')
for e in cons:
    print(f"  AU={e['au']:.3f}  BP={e['bp']:.3f}  n={e['n_members']}  "
          f"{', '.join(e['_members'][:3])}{'...' if e['n_members'] > 3 else ''}")

## What goes back to each project

| artifact | contents |
|---|---|
| `shared_features.csv` | the shared vocabulary |
| `federated_distance.csv` | pooled `p×p` |
| `federated_tree.json` | the pooled dendrogram |
| `federated_edges.csv` | per cluster: si, au, bp, se, v, c, pchi |

### And what a project does with them

**Not** shared patients — patients are disjoint across projects, so there is no
shared object set and patient clusters have nothing to pool. What federates is the
**feature space**: the analyte modules validated across every cohort's samples.

A project then clusters *its own* patients in that validated space. Endotypes stay
local; the space they are discovered in has the whole federation behind it. Cohort
demographics characterise the endotypes afterwards rather than driving them.

In [ ]:
# each cluster that survived federation becomes one module score per sample
modules = {e['edge_id']: e['_members'] for e in cons}
local = cohorts['cohort_A']
scores = pd.DataFrame({f'module_{i+1}': local[m].mean(axis=1)
                       for i, m in enumerate(modules.values())})
print(f'cohort_A: {scores.shape[0]} samples described by {scores.shape[1]} '
      f'federation-validated modules instead of {len(shared)} raw objects')
scores.head()

---
That reduced, validated matrix is where patient-level endotype clustering happens —
with `pvclust` or `kmeans_pv` from notebook 1, run locally.

**One caveat to carry forward.** Clustering patients means resampling analytes, and
analytes are co-expressed rather than independent, so AU there is anti-conservative.
It is a real number, but it is optimistic. Report it as such.